<a href="https://colab.research.google.com/github/raki-rankawat/thesis-v1/blob/master/Pipeline_Quantz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline_Quantz — Universal Export & Validation Pipeline

Exports every deployable checkpoint to FP32 ONNX + INT8 QDQ ONNX and validates on-device accuracy.

**Sources (auto-discovered — no manual list needed):**
- Seed baselines — hardcoded, always included
- Pruning checkpoints — loaded from `pruning_combined_records.json` (quant_ok = True only)
- KD checkpoints — loaded from `kd_combined_records.json` (quant_ok = True only)

**Per-run pipeline:**
1. Load checkpoint → evaluate FP32 val accuracy
2. Export FP32 ONNX → calibrate → INT8 QDQ ONNX
3. Compute NSE (FP32 vs INT8 logits)
4. Compute local INT8 accuracy on shared test set
5. Save record to `quantz_records.json` on Drive

**Resume support:** re-running skips already-exported runs automatically.  
STM32 on-device results are filled in separately after CLI runs.

**Never change** `SHARED_DIR` — all STM32 CLI commands must use the same `test_input.npz`.

In [ ]:
# ── One-time: fix stale nse_ok values after test-set correction ─────
# Corrects any record where nse_ok disagrees with the NSE value on disk.
import json as _json_fix
from pathlib import Path as _Path

for _jp in [Path(QUANTZ_RECORDS_JSON), Path(PRUNING_RECORDS_JSON)]:
    if not _jp.exists():
        print(f"  Not found: {_jp}"); continue
    _recs = _json_fix.loads(_jp.read_text())
    _changed = 0
    for _r in _recs:
        _nse = _r.get("NSE")
        if _nse is None:
            continue
        _correct = _nse >= NSE_THRESHOLD
        if _r.get("nse_ok") != _correct:
            _key = _r.get("name", _r.get("run_key", "?"))
            print(f"  FIXING {_key}: nse_ok {_r['nse_ok']} → {_correct}  (NSE={_nse:.4f})")
            _r["nse_ok"] = _correct
            _changed += 1
    if _changed:
        _jp.write_text(_json_fix.dumps(_recs, indent=2))
        print(f"  Saved {_jp.name} ({_changed} changes)")
    else:
        print(f"  {_jp.name}: all nse_ok values correct — no changes needed")

In [ ]:
# ── STM32 deployment metrics (Flash / RAM / MACs) ────────────────────
# Run stedgeai analyze for each passing model, then this cell reads results.
#
# CLI (run once per model outside Colab):
#   stedgeai analyze -m exports/<name>/model_int8_qdq.onnx \
#                    --target stm32h7xx \
#                    --output exports/<name>/analyze_int8.txt
#
# Results are saved to Drive and parsed here automatically.

import re as _re

def parse_stedgeai_analyze(txt_path):
    """Parse Flash, RAM, and MACC from stedgeai analyze output file."""
    if not Path(txt_path).exists():
        return None
    text = Path(txt_path).read_text()
    flash = _re.search(r"Flash\s+\|\s+([\d,]+)\s+bytes", text)
    ram   = _re.search(r"RAM\s+\|\s+([\d,]+)\s+bytes", text)
    macc  = _re.search(r"MACCs\s+\|\s+([\d,]+)", text)
    return {
        "flash_kb": int(flash.group(1).replace(",","")) / 1024 if flash else None,
        "ram_kb":   int(ram.group(1).replace(",",""))   / 1024 if ram   else None,
        "macc_k":   int(macc.group(1).replace(",",""))  / 1000 if macc  else None,
    }

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as _f:
        _qrecs = json.load(_f)

    print(f"{'Model':<42} {'Flash(KB)':>10} {'RAM(KB)':>10} {'MACCs(K)':>10}")
    print("-" * 75)
    _any_found = False
    for _r in _qrecs:
        if not _r.get("nse_ok"):
            continue
        _analyze_path = Path(_r["out_dir"]) / "analyze_int8.txt"
        _m = parse_stedgeai_analyze(_analyze_path)
        if _m:
            _any_found = True
            _fs = f"{_m['flash_kb']:.1f}" if _m["flash_kb"] else "—"
            _rs = f"{_m['ram_kb']:.1f}"   if _m["ram_kb"]   else "—"
            _ms = f"{_m['macc_k']:.1f}"   if _m["macc_k"]   else "—"
            print(f"  {_r['name']:<40} {_fs:>10} {_rs:>10} {_ms:>10}")
        else:
            print(f"  {_r['name']:<40}   pending — run stedgeai analyze first")
    if not _any_found:
        print("  No analyze results found yet. Run the CLI command above for each model.")
else:
    print(f"⚠️  {QUANTZ_RECORDS_JSON} not found — run the main pipeline first")

In [ ]:
# ── Mount Drive & load utils ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/stm32-thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")
    print("⚠️  Place the utils/ folder at: My Drive/stm32-thesis/utils/")

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import json, math, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

!pip -q install onnx onnxruntime onnxruntime-tools
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_static, QuantType, QuantFormat, CalibrationDataReader

from utils.dataset import prepare_dataset, get_loaders, get_test_loader
from utils.models  import VWW_MobileNetV2, VWW_MobileNetV3
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────
SAVE_DIR   = "/content/drive/My Drive/stm32-thesis/checkpoints"
EXPORT_DIR = Path("/content/drive/My Drive/stm32-thesis/exports")
SHARED_DIR = EXPORT_DIR / "shared"

# ── Source JSON records from upstream pipelines ───────────────────────
PRUNING_RECORDS_JSON = f"{SAVE_DIR}/pruning_combined_records.json"
KD_RECORDS_JSON      = f"{SAVE_DIR}/kd_combined_records.json"

# ── This pipeline's own records ───────────────────────────────────────
QUANTZ_RECORDS_JSON  = f"{SAVE_DIR}/quantz_records.json"

# ── NSE threshold (must match upstream pipelines) ─────────────────────
NSE_THRESHOLD = 0.95

# ── Seed baselines — always included regardless of upstream records ───
SEED_RUNS = [
    {"name": "mobilenetv2_seed_74", "arch": "mobilenetv2",
     "ckpt": f"{SAVE_DIR}/mobilenetv2_seed_74.pth"},
    {"name": "mobilenetv3_seed_74", "arch": "mobilenetv3",
     "ckpt": f"{SAVE_DIR}/mobilenetv3_seed_74.pth"},
]

print("✅ Config loaded")

In [ ]:
# ── One-time: delete stale test files after test-set fix ────────────
# Run this cell ONCE after first pulling the fixed notebook.
# Safe to re-run — skips silently if files don't exist.
import os
for _p in [SHARED_DIR / "test_input.npz", SHARED_DIR / "test_labels.npz"]:
    _p = Path(_p)
    if _p.exists():
        os.remove(_p)
        print(f"Deleted {_p}")
    else:
        print(f"Already absent: {_p}")
print("✅ Stale files cleared — re-run generate_shared_test_files next")

In [ ]:
# ── Quantisation helpers ─────────────────────────────────────────────
!pip -q install onnx onnxruntime onnxruntime-tools

from utils.quantz import (
    generate_shared_test_files,
    export_onnx,
    save_calib_npz,
    quantize_int8,
    compute_nse,
    compute_local_int8_accuracy,
    compute_stm32_accuracy,
)
print("✅ Quantisation helpers loaded from utils.quantz")

In [ ]:
# ── Model builder ────────────────────────────────────────────────────
def build_model(arch):
    if arch == "mobilenetv2": return VWW_MobileNetV2().to(device)
    if arch == "mobilenetv3": return VWW_MobileNetV3().to(device)
    raise ValueError(f"Unknown arch: {arch!r}")

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────
prepare_dataset()
train_loader, val_loader = get_loaders(batch_size=64)
test_loader              = get_test_loader(batch_size=1)
print("✅ Dataset ready")

In [ ]:
# ── Build run list from upstream JSON records ────────────────────────
# Auto-discovers deployable checkpoints from pruning + KD pipelines.
# Falls back gracefully if a JSON doesn't exist yet.

RUNS = []

# 1. Seed baselines — always included
for r in SEED_RUNS:
    RUNS.append({
        "name"   : r["name"],
        "arch"   : r["arch"],
        "ckpt"   : r["ckpt"],
        "source" : "baseline",
    })

# 2. Pruning checkpoints — quant_ok = True only
if os.path.exists(PRUNING_RECORDS_JSON):
    with open(PRUNING_RECORDS_JSON) as f:
        pruning_records = json.load(f)
    added = 0
    for r in pruning_records:
        if not r.get("quant_ok", False):
            continue
        model      = r["model"]
        prune_type = r["prune_type"]
        pct        = r["target_%"]
        arch       = "mobilenetv2" if "V2" in model else "mobilenetv3"
        name       = f"{arch}_{prune_type}_{pct}pct"
        ckpt       = r.get("ckpt", f"{SAVE_DIR}/{name}.pth")
        RUNS.append({
            "name"   : name,
            "arch"   : arch,
            "ckpt"   : ckpt,
            "source" : f"pruning ({prune_type} {pct}%)",
        })
        added += 1
    print(f"✅ Loaded {added} deployable pruning checkpoints from records")
else:
    print(f"⚠️  Pruning records not found: {PRUNING_RECORDS_JSON}")
    print(f"   Run Model_Pruning_Combined first, or add checkpoints manually below.")

# 3. KD checkpoints — quant_ok = True only
if os.path.exists(KD_RECORDS_JSON):
    with open(KD_RECORDS_JSON) as f:
        kd_records = json.load(f)
    added = 0
    for r in kd_records:
        if not r.get("quant_ok", False):
            continue
        run_key  = r["run_key"]
        student  = r["student"]
        arch     = "mobilenetv2" if "V2" in student else "mobilenetv3"
        ckpt     = r.get("ckpt", f"{SAVE_DIR}/{run_key}.pth")
        RUNS.append({
            "name"   : run_key,
            "arch"   : arch,
            "ckpt"   : ckpt,
            "source" : f"kd ({r['teacher']} → {student} {r['init']})",
        })
        added += 1
    print(f"✅ Loaded {added} deployable KD checkpoints from records")
else:
    print(f"⚠️  KD records not found: {KD_RECORDS_JSON}")
    print(f"   Run Model_KD_Combined first, or add checkpoints manually below.")

# 4. Manual overrides — add extra checkpoints here if needed
# RUNS.append({
#     "name"   : "my_custom_model",
#     "arch"   : "mobilenetv2",
#     "ckpt"   : f"{SAVE_DIR}/my_custom_model.pth",
#     "source" : "manual",
# })

# Create export dirs
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
for r in RUNS:
    r["out_dir"] = EXPORT_DIR / r["name"]
    r["out_dir"].mkdir(parents=True, exist_ok=True)

# Pre-flight
print(f"\n{'─'*70}")
print(f"Total runs: {len(RUNS)}")
print(f"{'─'*70}")
for r in RUNS:
    ok = "✅" if Path(r["ckpt"]).exists() else "❌ MISSING"
    print(f"  {ok}  {r['name']:<40}  [{r['source']}]")

missing = [r for r in RUNS if not Path(r["ckpt"]).exists()]
if missing:
    print(f"\n⚠️  {len(missing)} checkpoint(s) missing — those runs will be skipped.")

In [ ]:
# ── Shared test files — generated once, reused by all runs ───────────
print("[Shared test set]")
SHARED_INPUT_NPZ, SHARED_LABELS_NPZ = generate_shared_test_files(SHARED_DIR, test_loader)
print(f"  Input : {SHARED_INPUT_NPZ}")
print(f"  Labels: {SHARED_LABELS_NPZ}")

In [ ]:
# ── Main export + validation loop ───────────────────────────────────
# Resume: loads existing records, skips completed runs.
# Saves progress after every run — safe to interrupt and re-run.

# ── Load existing progress ────────────────────────────────────────────
if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)
    print(f"✅ Loaded {len(records)} existing records")
    for r in records:
        print(f"   ↩️  {r['name']:<40}  NSE: {r['NSE']}  "
              f"local_int8: {r['local_int8_acc_%']}%")
else:
    records = []
    print("No existing records — starting fresh")

print()

def _already_done(name):
    return any(r["name"] == name for r in records)

for run in RUNS:
    name    = run["name"]
    arch    = run["arch"]
    ckpt    = run["ckpt"]
    out_dir = run["out_dir"]
    source  = run["source"]

    # ── Skip if already exported ──────────────────────────────────────
    if _already_done(name):
        print(f"⏭️  {name} — already done, skipping")
        continue

    print(f"\n{'═'*65}")
    print(f"  {name}  [{source}]")
    print(f"{'═'*65}")

    if not Path(ckpt).exists():
        print(f"  ❌ Checkpoint missing, skipping: {ckpt}")
        continue

    # ── [1] Load + eval ───────────────────────────────────────────────
    model = build_model(arch)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    model.eval()
    val_acc = evaluate(model, val_loader, device)
    print(f"  FP32 val acc : {val_acc*100:.2f}%")

    # ── [2] FP32 ONNX ─────────────────────────────────────────────────
    fp32_path  = out_dir / "model_fp32.onnx"
    calib_path = out_dir / "calib_train.npz"
    int8_path  = out_dir / "model_int8_qdq.onnx"

    export_onnx(model, fp32_path, device)
    save_calib_npz(calib_path, train_loader)

    # ── [3] INT8 QDQ ──────────────────────────────────────────────────
    try:
        quantize_int8(fp32_path, calib_path, int8_path)
        nse           = compute_nse(fp32_path, int8_path, SHARED_INPUT_NPZ)
        local_int8    = compute_local_int8_accuracy(int8_path,
                                                    SHARED_INPUT_NPZ,
                                                    SHARED_LABELS_NPZ)
        nse_ok        = nse >= NSE_THRESHOLD
        quant_error   = None
    except Exception as e:
        nse         = float("nan")
        local_int8  = float("nan")
        nse_ok      = False
        quant_error = str(e)
        print(f"    ⚠️  Quantisation failed: {e}")

    gate = "✅ deploy" if nse_ok else "⛔ quant failed"
    print(f"  NSE          : {nse:.4f}  {gate}")
    print(f"  Local INT8   : {local_int8:.2f}%  "
          f"(200-sample estimate, not comparable to FP32 val)")

    records.append({
        "name"            : name,
        "arch"            : arch,
        "source"          : source,
        "fp32_val_acc_%"  : round(val_acc * 100, 2),
        "local_int8_acc_%": round(local_int8, 2) if not math.isnan(local_int8) else None,
        "NSE"             : round(nse, 4) if not math.isnan(nse) else None,
        "nse_ok"          : nse_ok,
        "quant_error"     : quant_error,
        "fp32_path"       : str(fp32_path),
        "int8_path"       : str(int8_path) if nse_ok else None,
        "out_dir"         : str(out_dir),
        "stm32_fp32_acc_%": None,   # filled in by STM32 cell
        "stm32_int8_acc_%": None,   # filled in by STM32 cell
    })

    # ── Save progress after every run ─────────────────────────────────
    with open(QUANTZ_RECORDS_JSON, "w") as f:
        json.dump(records, f, indent=2)
    print(f"  💾 Progress saved ({len(records)} records total)")

    del model

print("\n\n✅ Export loop complete.")
print(f"   Records JSON: {QUANTZ_RECORDS_JSON}")

In [ ]:
# ── Results table ────────────────────────────────────────────────────
# Reloads from JSON — works independently across sessions.

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)

df = pd.DataFrame(records)

W = 100
print("=" * W)
print("PIPELINE_QUANTZ — CONSOLIDATED RESULTS")
print("=" * W)

cols = ["name", "source", "fp32_val_acc_%", "local_int8_acc_%", "NSE", "nse_ok"]
# Note: local_int8_acc_% is on 200 calibration samples, fp32_val_acc_% is on 1500 val samples
# — these are different sets so no delta is shown. Use STM32 cell for apples-to-apples INT8 comparison.
print(df[cols].to_string(index=False))
print("=" * W)

# ── NSE summary ───────────────────────────────────────────────────────
print("\nNSE gate summary:")
for _, r in df.iterrows():
    gate    = "✅ PASS" if r["nse_ok"] else "⛔ FAIL"
    nse_str = f"{r['NSE']:.4f}" if r["NSE"] is not None else "ERROR"
    print(f"  {r['name']:<42}  NSE={nse_str}  {gate}")

# ── STM32 results (if available) ──────────────────────────────────────
stm32_cols = ["stm32_fp32_acc_%", "stm32_int8_acc_%"]
has_stm32  = df[stm32_cols].notna().any().any()
if has_stm32:
    print(f"\nSTM32 on-device results:")
    print(f"  {'Name':<42} {'FP32':>8}  {'INT8':>8}  {'Delta':>8}")
    print(f"  {'─'*72}")
    for _, r in df.iterrows():
        fp32 = f"{r['stm32_fp32_acc_%']:.2f}%" if r["stm32_fp32_acc_%"] is not None else "pending"
        int8 = f"{r['stm32_int8_acc_%']:.2f}%" if r["stm32_int8_acc_%"] is not None else "pending"
        if r["stm32_fp32_acc_%"] and r["stm32_int8_acc_%"]:
            delta = f"{r['stm32_int8_acc_%'] - r['stm32_fp32_acc_%']:+.2f}%"
        else:
            delta = "—"
        print(f"  {r['name']:<42} {fp32:>8}  {int8:>8}  {delta:>8}")
else:
    print("\nSTM32 on-device results: not yet available (run STM32 CLI, then Cell 12)")

## STM32 CLI Commands

All models use the **same** `test_input.npz` from `exports/shared/`. Never change this path.

```bash
# Template — replace <name> with the run name (e.g. mobilenetv2_seed_74)

# FP32
stedgeai validate -m exports/<name>/model_fp32.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_fp32.npz

# INT8
stedgeai validate -m exports/<name>/model_int8_qdq.onnx \
                  --target stm32n6 --mode target \
                  --valinput exports/shared/test_input.npz
cp network_val_io.npz exports/<name>/output_int8.npz
```

Run the cell below after copying `.npz` outputs back to Drive.

In [ ]:
# ── STM32 on-device results ──────────────────────────────────────────
# Run this cell after STM32 CLI output .npz files are copied back to Drive.
# Updates quantz_records.json with on-device accuracy — safe to re-run
# as results come in incrementally.

if os.path.exists(QUANTZ_RECORDS_JSON):
    with open(QUANTZ_RECORDS_JSON) as f:
        records = json.load(f)

updated = 0
for r in records:
    out_dir   = Path(r["out_dir"])
    fp32_out  = out_dir / "output_fp32.npz"
    int8_out  = out_dir / "output_int8.npz"

    changed = False

    if fp32_out.exists() and r["stm32_fp32_acc_%"] is None:
        try:
            acc = compute_stm32_accuracy(SHARED_LABELS_NPZ, fp32_out)
            r["stm32_fp32_acc_%"] = round(acc, 2)
            print(f"  ✅ {r['name']:<42}  STM32 FP32: {acc:.2f}%")
            changed = True
        except Exception as e:
            print(f"  ⚠️  {r['name']} FP32 error: {e}")

    if int8_out.exists() and r["stm32_int8_acc_%"] is None:
        try:
            acc = compute_stm32_accuracy(SHARED_LABELS_NPZ, int8_out)
            r["stm32_int8_acc_%"] = round(acc, 2)
            print(f"  ✅ {r['name']:<42}  STM32 INT8: {acc:.2f}%")
            changed = True
        except Exception as e:
            print(f"  ⚠️  {r['name']} INT8 error: {e}")

    if changed:
        updated += 1

if updated > 0:
    with open(QUANTZ_RECORDS_JSON, "w") as f:
        json.dump(records, f, indent=2)
    print(f"\n💾 Saved {updated} updated records to {QUANTZ_RECORDS_JSON}")
else:
    print("No new STM32 results found. Copy output_fp32.npz / output_int8.npz")
    print("to their respective export dirs and re-run this cell.")

# Print full STM32 table
print(f"\n{'─'*80}")
print(f"{'Name':<42} {'FP32 val':>9} {'STM32 FP32':>11} {'STM32 INT8':>11} {'Delta':>8}")
print(f"{'─'*80}")
for r in records:
    fp32_stm = f"{r['stm32_fp32_acc_%']:.2f}%" if r["stm32_fp32_acc_%"] is not None else "pending"
    int8_stm = f"{r['stm32_int8_acc_%']:.2f}%" if r["stm32_int8_acc_%"] is not None else "pending"
    if r["stm32_fp32_acc_%"] and r["stm32_int8_acc_%"]:
        delta = f"{r['stm32_int8_acc_%'] - r['stm32_fp32_acc_%']:+.2f}%"
    else:
        delta = "—"
    print(f"{r['name']:<42} {r['fp32_val_acc_%']:>8.2f}% {fp32_stm:>11} {int8_stm:>11} {delta:>8}")

## Resume / re-run guide

**Add a new checkpoint:** just re-run `Model_Pruning_Combined` or `Model_KD_Combined` with the new experiment — the deployable list updates automatically in `pruning_combined_records.json` / `kd_combined_records.json`. Then re-run the build run list cell here and the export loop will pick up the new entry.

**Force re-export of one run:** delete its entry from `quantz_records.json` on Drive, delete its export folder under `exports/<name>/`, then re-run the export loop.

**Force everything from scratch:** delete `quantz_records.json` and re-run.

**STM32 results come in incrementally:** just copy each `output_fp32.npz` / `output_int8.npz` back to the right `exports/<name>/` folder and re-run Cell 12 — it only updates records that don't already have on-device results.